# 🛡️ DistilBERT Classifier Retraining & Evaluation Bundle (Colab GPU)
**Project:** Prompt-Injection Benchmark & Lightweight Defense for Tool-Using LLM Agents  
**Protocol:** Publication-Grade Cluster-Isolated Splits (Train 8,179 / Cal 1,752 / Test 1,752)  
**Architecture:** `distilbert-base-uncased` (FP32, Pinned Seed 42, 0 NaN Guarantee)  
**Hardware:** Google Colab Free T4 GPU

### Step 1: Verify Free T4 GPU Environment
In Colab menu: **Runtime** ➔ **Change runtime type** ➔ select **T4 GPU** ➔ **Save**.

In [ ]:
# 1. Upgrade sympy and install dependencies cleanly
!pip install -q --upgrade sympy
!pip install -q transformers datasets accelerate scikit-learn evaluate pandas pyarrow safetensors
import torch, os, random, hashlib, json, gc
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name :', torch.cuda.get_device_name(0))


### Step 2: Upload or Fetch Cluster-Isolated Splits
Upload `train.csv`, `calibration.csv`, `test.csv` (from your local `data/splits/`), and `benchmark_cases.jsonl`.

In [ ]:
import glob, os, pandas as pd
from google.colab import files

def get_file(prefix, ext):
    matches = sorted(glob.glob(f'{prefix}*.{ext}') + glob.glob(f'/content/{prefix}*.{ext}'))
    return matches[0] if matches else None

train_f = get_file('train', 'csv')
cal_f = get_file('calibration', 'csv')
test_f = get_file('test', 'csv')
bench_f = get_file('benchmark_cases', 'jsonl')

if not (train_f and cal_f and test_f and bench_f):
    print('Files missing in Colab session. Please upload the 4 files:')
    print('1. train.csv (data/splits/)')
    print('2. calibration.csv (data/splits/)')
    print('3. test.csv (data/splits/)')
    print('4. benchmark_cases.jsonl (data/)')
    files.upload()
    train_f = get_file('train', 'csv')
    cal_f = get_file('calibration', 'csv')
    test_f = get_file('test', 'csv')
    bench_f = get_file('benchmark_cases', 'jsonl')

train_df = pd.read_csv(train_f)
cal_df = pd.read_csv(cal_f)
test_df = pd.read_csv(test_f)
print(f'Splits loaded successfully: Train={len(train_df)}, Cal={len(cal_df)}, Test={len(test_df)}')
print(f'Using files: train={train_f}, cal={cal_f}, test={test_f}, bench={bench_f}')


### Step 3: Train Primary DistilBERT Classifier (OOM-Safe Dynamic Padding)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import gc, torch

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

train_ds = Dataset.from_dict({'text': train_df['text'].tolist(), 'label': train_df['label'].tolist()})
cal_ds = Dataset.from_dict({'text': cal_df['text'].tolist(), 'label': cal_df['label'].tolist()})

# Dynamic padding: tokenize without padding to max_length; DataCollator handles batch-level padding
def tokenize_fn(b):
    return tokenizer(b['text'], truncation=True, max_length=512)

train_tok = train_ds.map(tokenize_fn, batched=True, batch_size=1000)
cal_tok = cal_ds.map(tokenize_fn, batched=True, batch_size=1000)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# OOM-Safe Training Arguments: save_strategy='no' prevents Linux OOM crash during shard serialization
training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy='no',          # Prevents OOM crash from saving intermediate model shards
    eval_strategy='no',          # Prevents RAM spikes during training; evaluated cleanly in Step 5
    seed=SEED,
    fp16=False,                  # FP32 strictly prevents NaN gradients
    dataloader_num_workers=0,    # Avoids shared memory crash in Colab
    report_to='none',
)

trainer_kwargs = {'model': model, 'args': training_args, 'train_dataset': train_tok, 'data_collator': data_collator}
import inspect
if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = Trainer(**trainer_kwargs)

print('Training primary DistilBERT classifier on T4 GPU (OOM-Safe Fast Mode)...')
trainer.train()

# Garbage collect and save model cleanly
gc.collect()
torch.cuda.empty_cache()

SAVE_DIR = './injection_classifier'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Primary model saved cleanly to', SAVE_DIR)


### Step 4: Verify Weights & Compute Checkpoint SHA-256

In [ ]:
safetensor_path = os.path.join(SAVE_DIR, 'model.safetensors')
hasher = hashlib.sha256()
with open(safetensor_path, 'rb') as f:
    while chunk := f.read(65536):
        hasher.update(chunk)
checkpoint_sha256 = hasher.hexdigest()
print('Model Checkpoint SHA-256:', checkpoint_sha256)

# Check for NaNs
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print('WARNING: NaN detected in tensor:', name)
        has_nan = True
if not has_nan:
    print('VERIFIED: 0 NaN weights across all tensors!')


### Step 5: Export Evaluation Scores (Calibration, Test, Benchmark)

In [ ]:
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def predict_probs(texts, batch_size=64):
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
            logits = model(**inputs).logits
            p = torch.softmax(logits, dim=-1)[:, 1].cpu().tolist()
            probs.extend(p)
    return probs

print('Predicting calibration probabilities...')
cal_probs = predict_probs(cal_df['text'].tolist())
with open('scores_calibration.json', 'w') as f:
    json.dump({'model_name': 'distilbert_retrained', 'checkpoint_sha256': checkpoint_sha256, 'probabilities': cal_probs}, f)

print('Predicting test probabilities...')
test_probs = predict_probs(test_df['text'].tolist())
with open('scores_test.json', 'w') as f:
    json.dump({'model_name': 'distilbert_retrained', 'checkpoint_sha256': checkpoint_sha256, 'probabilities': test_probs}, f)

# Benchmark predictions
bench_cases = [json.loads(l) for l in open(bench_f, encoding='utf-8') if l.strip()]
bench_texts = [c.get('user_task', '') if c.get('is_benign') else c.get('injection_payload', '') for c in bench_cases]
bench_probs = predict_probs(bench_texts)
with open('scores_benchmark.json', 'w') as f:
    json.dump({'model_name': 'distilbert_retrained', 'checkpoint_sha256': checkpoint_sha256, 'probabilities': bench_probs}, f)

print('Score files exported successfully!')


### Step 6: Train LOSO Models (S-Labs ➔ deepset)

In [ ]:
# LOSO 1: Train on S-Labs only, score deepset test
slabs_train = train_df[train_df['source'] == 'slabs']
deepset_test = test_df[test_df['source'] == 'deepset']

model_loso1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
slabs_ds = Dataset.from_dict({'text': slabs_train['text'].tolist(), 'label': slabs_train['label'].tolist()}).map(tokenize_fn, batched=True, batch_size=1000)

args_loso1 = TrainingArguments(
    output_dir='./ck_loso1',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    save_strategy='no',
    eval_strategy='no',
    seed=SEED,
    fp16=False,
    dataloader_num_workers=0,
    report_to='none',
)
loso_kwargs = {'model': model_loso1, 'args': args_loso1, 'train_dataset': slabs_ds, 'data_collator': data_collator}
if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
    loso_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in inspect.signature(Trainer.__init__).parameters:
    loso_kwargs['tokenizer'] = tokenizer
trainer_loso1 = Trainer(**loso_kwargs)
print('Training LOSO 1 (S-Labs -> deepset)...')
trainer_loso1.train()

gc.collect()
torch.cuda.empty_cache()

model_loso1.eval()
model_loso1.to(device)

def predict_loso1(texts):
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            inputs = tokenizer(texts[i:i+64], return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
            logits = model_loso1(**inputs).logits
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
    return probs

loso1_probs = predict_loso1(deepset_test['text'].tolist())
with open('scores_loso_slabs_to_deepset.json', 'w') as f:
    json.dump({'probabilities': loso1_probs}, f)
print('LOSO 1 scores exported!')


### Step 7: Bundle Everything for Download

In [ ]:
from google.colab import files
!zip -r classifier_eval_bundle.zip ./injection_classifier scores_calibration.json scores_test.json scores_benchmark.json scores_loso_slabs_to_deepset.json
print('Bundle created: classifier_eval_bundle.zip')
files.download('classifier_eval_bundle.zip')
